# Quantization Troubleshooting with the Model Compression Toolkit (MCT) Using the XQuant Extension Tool (General Troubleshooting)

[Run this tutorial in Google Colab](https://colab.research.google.com/github/SonySemiconductorSolutions/mct-model-optimization/blob/main/tutorials/notebooks/mct_features_notebooks/pytorch/example_pytorch_XQuant_Extension_Tool_General.ipynb)

## Overview
This notebook provides general troubleshooting actions and practical guidance for improving the quality of Post-Training Quantization of PyTorch models.

Judgeable troubleshooting actions based on the analysis results report from the XQuant extension tool are listed [here](https://colab.research.google.com/github/SonySemiconductorSolutions/mct-model-optimization/blob/main/tutorials/notebooks/mct_features_notebooks/pytorch/example_pytorch_XQuant_Extension_Tool.ipynb)

## Summary
We will cover the following steps:

1. MobileNet V3 Setting
2. Dataset Preparation
3. Perform Post-Training Quantization using MCT (default parameter)
4. General Troubleshooting
   - Representative Dataset Size & Diversity
   - Bias Correction
   - Using More Samples in Mixed Precision Quantization
   - Threshold Selection Error Method
   - Enabling Hessian-Based Mixed Precision
   - GPTQ - Gradient-Based Post-Training Quantization
5. Conclusion

## Setup
Install the relevant packages:

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0

In [ ]:
import importlib
if not importlib.util.find_spec('model_compression_toolkit'):
    !pip install model_compression_toolkit

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torchvision.datasets import ImageNet
import numpy as np
import random
import model_compression_toolkit as mct
from tqdm import tqdm
import os
import gc

## MobileNet V3 Setting
We will begin by quantizing MobileNetV3 using the `pytorch_post_training_quantization`  function from MCT.

In [ ]:
weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2
float_model = mobilenet_v3_large(weights=weights)

## Dataset Preparation
### Download ImageNet validation set
Download ImageNet dataset (validation split only).

This step may take several minutes...

**Note:** For demonstration purposes, we use the validation set for the model quantization routines. Usually, a subset of the training dataset is used, but loading it is a heavy procedure that is unnecessary for the sake of this demonstration.

In [ ]:
if not os.path.isdir('imagenet'):
    !mkdir imagenet
    !wget -P imagenet https://image-net.org/data/ILSVRC/2012/ILSVRC2012_devkit_t12.tar.gz
    !wget -P imagenet https://image-net.org/data/ILSVRC/2012/ILSVRC2012_img_val.tar

Extract ImageNet validation dataset using torchvision "datasets" module.

In [ ]:
dataset = ImageNet(root='./imagenet', split='val', transform=weights.transforms())

## Representative Dataset
For quantization with MCT, we need to define a representative dataset. This dataset is a generator that returns a list of images:

In [ ]:
batch_size = 16
n_iter = 10

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

def representative_dataset_gen():
    dataloader_iter = iter(dataloader)
    for _ in range(n_iter):
        yield [next(dataloader_iter)[0]]

In [ ]:
def make_dataloader_and_reset_random_seed():
    seed = 0

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(seed)

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, worker_init_fn=seed_worker, generator=g)

    return dataloader

## Evaluation Function
Evaluate the model's performance using the validation data loader and calculate and return the overall classification accuracy.

In [ ]:
def evaluate(model, val_dataloader):
    """
    Evaluate a model using a validation loader.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for data in tqdm(val_dataloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_acc = (100 * correct / total)
    print('Accuracy: %.2f%%' % val_acc)
    return val_acc

## Perform Post-Training Quantization using MCT (default parameter)
Run Post-Training Quantization for MCT with default settings.

In [ ]:
# Get a TargetPlatformCapabilities object that models the hardware platform for the quantized model inference. 
target_platform_cap = mct.get_target_platform_capabilities(tpc_version=1.0)
configuration = mct.core.CoreConfig()
dataloader = make_dataloader_and_reset_random_seed()

# Post-Training Quantization using MCT
quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                                        representative_data_gen=representative_dataset_gen,
                                                                                        core_config = configuration,
                                                                                        target_platform_capabilities=target_platform_cap
                                                                                        )

## Model Evaluation
We evaluated the model before and after quantization using the same dataset.

In [ ]:
val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
evaluate(float_model, val_dataloader)
evaluate(quantized_model, val_dataloader)
torch.cuda.empty_cache()
del quantized_model
gc.collect()

The accuracy of quantized_model was 73%, 2.27% lower than float_model. We will improve this.

## General Troubleshooting
If there is no significant improvement, comprehensively evaluate other areas for improvement.
The following items are general troubleshoots for quantization accuracy improvement.

### Representative Dataset Size & Diversity
The representative dataset is used by MCT to derive the threshold for the model's activation tensor.
If the representative dataset is too small or not diverse enough, accuracy may decrease.
Increase the number of samples in the representative dataset or increase the diversity of the samples.

See [Troubleshooting Documentation >> Representative Dataset Size & Diversity](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/representative_dataset_size_and_diversity.html#ug-representative-dataset-size-and-diversity)

In [ ]:
# Change representative dataset and re-run the quantization process.
number_of_iter = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
results = []
for niter in number_of_iter:
    print(f"Number of iterations for representative dataset: {niter}")
    def representative_dataset_gen2():
        dataloader_iter = iter(dataloader)
        for _ in range(niter):
            yield [next(dataloader_iter)[0]]

    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()

    # Post-Training Quantization using MCT
    quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                    representative_data_gen=representative_dataset_gen2,
                                                                    core_config = configuration,
                                                                    target_platform_capabilities=target_platform_cap
                                                                    )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()

In [ ]:
# Print results after each iteration
for num_img, acc in zip(number_of_iter, results):
    print(f"Number of Images: {num_img}, Accuracy: {acc:.2f}%")

### Bias Correction
MCT applies bias correction by default to overcome the induced bias shift caused by weights quantization.

You can check if the bias correction causes a degradation in accuracy by disabling the bias correction (setting weights_bias_correction to False in the QuantizationConfig of CoreConfig).

See [Troubleshooting Documentation >> Bias Correction](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/bias_correction.html#ug-bias-correction)

In [ ]:
true_or_false = [True, False]
results = []
for flag in true_or_false:
    print(f"Weight and bias correction enabled: {flag}")
    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()

    configuration = mct.core.CoreConfig(mct.core.QuantizationConfig(weights_bias_correction=flag))

    # Post-Training Quantization using MCT
    quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                    representative_data_gen=representative_dataset_gen,
                                                                    core_config = configuration,
                                                                    target_platform_capabilities=target_platform_cap
                                                                    )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()

In [ ]:
# Print results after each iteration
for flag, acc in zip(true_or_false, results):
    print(f"Bias Correction Enabled: {flag}, Accuracy: {acc:.2f}%")

### Using More Samples in Mixed Precision Quantization

In Mixed Precision quantization, MCT will assign a different bit width to each weight in the model, depending on the weight's layer sensitivity and a resource constraint defined by the user, such as target model size.

By default, MCT employs 32 samples from the provided representative dataset for the Mixed Precision search. Leveraging a larger dataset could enhance results, particularly when dealing with datasets exhibiting high variance.

Set the num_of_images attribute to a larger value of the MixedPrecisionQuantizationConfig in CoreConfig.

See [Troubleshooting Documentation >> Using More Samples in Mixed Precision Quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/using_more_samples_in_mixed_precision_quantization.html#ug-using-more-samples-in-mixed-precision-quantization)

In [ ]:
number_of_image = [32, 64]
results = []
for NumOfIg in number_of_image:
    print(f"Number of images for Mixed Precision quantization: {NumOfIg}")
            
    # Add troubleshooting items to this Config (Mixed Precision)
    mixed_precision_config = mct.core.MixedPrecisionQuantizationConfig(num_of_images=NumOfIg)
    configuration = mct.core.CoreConfig(mixed_precision_config=mixed_precision_config)

    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()

    # Post-Training Quantization using MCT
    quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                    representative_data_gen=representative_dataset_gen,
                                                                    core_config = configuration,
                                                                    target_platform_capabilities=target_platform_cap
                                                                    )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()

In [ ]:
# Print results after each iteration
for num_img, acc in zip(number_of_image, results):
    print(f"Number of Images: {num_img}, Accuracy: {acc:.2f}%")

### Threshold Selection Error Method
MCT defaults to employing the Mean-Squared Error (MSE) metric for threshold optimization, however, it offers a range of alternative error metrics (e.g. using min/max values, KL-divergence, etc.) to accommodate different network requirements.

We advise you to consider other error metrics if your model is suffering from significant accuracy degradation, especially if it contains unorthodox activation layers.

For example, set NOCLIPPING to the activation_error_method attribute of the QuantizationConfig in CoreConfig.

See [Troubleshooting Documentation >> Threshold Selection Error Method](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/threhold_selection_error_method.html#ug-threshold-selection-error-method)

In [ ]:
threshold_selection_error_method = [mct.core.QuantizationErrorMethod.NOCLIPPING,
                                    mct.core.QuantizationErrorMethod.MSE,
                                    mct.core.QuantizationErrorMethod.MAE,
                                    mct.core.QuantizationErrorMethod.KL,
                                    mct.core.QuantizationErrorMethod.LP
                                    ]
results = []
for method in threshold_selection_error_method:
    print(f"Threshold selection error method: {method}")

    # Add troubleshooting items to this Config
    configuration = mct.core.CoreConfig(mct.core.QuantizationConfig(activation_error_method=method))

    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()

    # Post-Training Quantization using MCT
    quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                    representative_data_gen=representative_dataset_gen,
                                                                    core_config = configuration,
                                                                    target_platform_capabilities=target_platform_cap
                                                                    )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()

In [ ]:
# Print results after each iteration
for method, acc in zip(threshold_selection_error_method, results):
    print(f"Error Method: {method}, Accuracy: {acc:.2f}%")

### Enabling Hessian-Based Mixed Precision
MCT offers a Hessian-Based scoring mechanism to assess the importance of layers during the Mixed Precision search.
This feature can notably enhance Mixed Precision outcomes for certain network architectures.

Set the use_hessian_based_scores flag to True in the MixedPrecisionQuantizationConfig of the CoreConfig.

See [Troubleshooting Documentation >> Enabling Hessian-Based Mixed Precision](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/enabling_hessian-based_mixed_precision.html#ug-enabling-hessian-based-mixed-precision)


In [ ]:
true_or_false = [True, False]
results = []
for flag in true_or_false:
    print(f"Hessian-based Mixed precision scores enabled: {flag}")
    
    # Add troubleshooting items to this Config (Mixed Precision)
    mixed_precision_config = mct.core.MixedPrecisionQuantizationConfig(use_hessian_based_scores=flag)
    configuration = mct.core.CoreConfig(mixed_precision_config=mixed_precision_config)

    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()

    # Post-Training Quantization using MCT
    quantized_model, _ = mct.ptq.pytorch_post_training_quantization(in_module=float_model,
                                                                    representative_data_gen=representative_dataset_gen,
                                                                    core_config = configuration,
                                                                    target_platform_capabilities=target_platform_cap
                                                                    )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()    

In [ ]:
# Print results after each iteration
for flag, acc in zip(true_or_false, results):
    print(f"Hessian-based Scores Enabled: {flag}, Accuracy: {acc:.2f}%") 

### GPTQ - Gradient-Based Post-Training Quantization
When PTQ (either with or without Mixed Precision) fails to deliver the required accuracy, GPTQ is potentially the remedy.

MCT can configure GPTQ optimization options, such as the number of epochs for the optimization process.

See [Troubleshooting Documentation >> GPTQ - Gradient-Based Post-Training Quantization](https://sonysemiconductorsolutions.github.io/mct-model-optimization/docs_troubleshoot/troubleshoots/gptq-gradient_based_post_training_quantization.html#ug-gptq-gradient-based-post-training-quantization)

In [ ]:
gptq_arg = [60, 70, 80, 90]
results = []
for gptq in gptq_arg:
    print(f"Number of epochs for GPTQ: {gptq}")
    # Add random seed processing
    dataloader = make_dataloader_and_reset_random_seed()
    gptq_config = mct.gptq.get_pytorch_gptq_config(n_epochs=gptq)
    quantized_model, _ = mct.gptq.pytorch_gradient_post_training_quantization(float_model,
                                                                              representative_dataset_gen,
                                                                              gptq_config=gptq_config,
                                                                              target_platform_capabilities=target_platform_cap
                                                                              )
    # Model Evaluation
    val_dataloader = DataLoader(dataset, batch_size=50, shuffle=False, num_workers=16, pin_memory=True)
    val_acc = evaluate(quantized_model, val_dataloader)
    results.append(val_acc)
    torch.cuda.empty_cache()
    del quantized_model
    gc.collect()

In [ ]:
# Print results after each iteration
for epochs, acc in zip(gptq_arg, results):
    print(f"GPTQ Epochs: {epochs}, Accuracy: {acc:.2f}%")

## Conclusion
These analyses showed that accuracy improved by 0.31% when the number of images was 80, and by 0.54% when the number of GPTQ epochs was 80, resulting in a reduced quantization accuracy loss. These troubleshooting steps can help improve the accuracy of your quantized model.

## Copyrights
Copyright 2026 Sony Semiconductor Solutions, Inc. All rights reserved.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
